# OpenJev: zero-shot và few-shot inference

Chọn MODE để chạy checkpoint OpenJev Flash 9B gốc bằng **zero_shot** hoặc **few_shot**. Cả hai giữ nguyên weights BF16, không fine-tune, không LoRA và không boosting. Cùng dùng câu hỏi, logits A/B, cách ghép target và metrics như notebook train.

Few-shot đưa ví dụ từ train vào lịch sử user/assistant trước câu hỏi cần dự đoán. Với mỗi câu hỏi, cố gắng cân bằng yes/no hoặc individual/group; không lấy ví dụ trùng ID, comment hoặc video với DATA/DEV, không dùng mẫu tổng hợp. SHOTS là số ví dụ mỗi câu hỏi, không phải mỗi identity được chọn. Nhãn thiếu mẫu dương sẽ được báo rõ.

Chọn GPU trong runtime, upload TSV vào session, rồi sửa đường dẫn. Không mount Drive; không cần GPU local. Colab clone main vào /content/semeval; Kaggle dùng /kaggle/working/semeval (bật GPU và Internet).

## 1. Lấy code và cài thư viện
Chạy phần này trước khi import torch/transformers. Nếu session đã import các thư viện đó trước khi cài, khởi động lại session rồi chạy từ đầu.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = "https://github.com/iamtro-2006/semeval.git"
BRANCH = "main"
BASE = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
REPO_DIR = BASE / "semeval"
BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)
else:
    current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    changes = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
    if current != BRANCH or changes:
        raise RuntimeError("Existing clone has a different branch or local edits. Choose a new REPO_DIR.")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

WORK = REPO_DIR / "task 8" / "openjev"
assert (WORK / "run.py").is_file(), WORK
print("Code:", WORK)
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)


In [ ]:
import importlib.metadata
from packaging.version import Version

packages = ["transformers==5.17.0", "accelerate==1.13.0",
            "safetensors>=0.7,<1", "tqdm>=4.67,<5"]
try:
    torch_version = Version(importlib.metadata.version("torch").split("+")[0])
except importlib.metadata.PackageNotFoundError:
    torch_version = Version("0")
# Preserve a recent Colab/Kaggle PyTorch stack. Upgrade the matched trio only if needed.
if torch_version < Version("2.8"):
    packages += ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import importlib.util
spec = importlib.util.spec_from_file_location("openjev_run", WORK / "run.py")
run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run)
print("Ready. Model weights have not been downloaded yet.")


## 2. Chọn chế độ và dataset
Dùng các file **gốc trong data/split/en** của máy local, upload trực tiếp vào session:

| File | Số mẫu hiện tại | Vai trò |
|---|---:|---|
| StereoQueerEval_EN_train.tsv | 1.107 | Nguồn ví dụ cho few-shot; zero-shot không cần |
| StereoQueerEval_EN_val.tsv | 316 | DATA khi phát triển/thử nghiệm |
| StereoQueerEval_EN_test.tsv | 159 | DATA cho đánh giá cuối sau khi chốt cấu hình |

Không dùng raw training hoặc các bản augmented đã tạo trước đây để thay thế các split gốc. Không cần tạo lại split hoặc sửa TSV. Chỉ sửa đường dẫn nếu bạn đặt tên file khác.

DATA cần các cột StereoQueerEval_id, yt_title, yt_description, yt_comment. Cột target tùy chọn cho DATA; có nhãn thì tính metrics. TRAIN cho few-shot phải có target. Dev tổng hợp không được trộn vào điểm dev gốc. File samples.json chỉ phục vụ fine-tune; notebook này không đọc nó.

Mặc định THRESHOLD=0.5, TUNE_ON_DEV=False. Khi chạy thử trên val, giữ DEV trống. Nếu muốn chọn ngưỡng trên dev để đánh giá cuối: đặt DATA=test, DEV=val và TUNE_ON_DEV=True. Kết quả khi đó được ghi là zero_shot_dev_threshold hoặc few_shot_dev_threshold. Không dùng test để chọn ngưỡng.

Dùng cùng DATA và chính sách ngưỡng để so sánh hai chế độ; OUTPUT mặc định tách theo MODE.


In [ ]:
MODE = "zero_shot"  # "zero_shot" or "few_shot"
DATA = str(BASE / "StereoQueerEval_EN_val.tsv")
TRAIN = str(BASE / "StereoQueerEval_EN_train.tsv")  # Only needed for few_shot
SHOTS = 4  # Demonstrations per question
OUTPUT = str(BASE / MODE.replace("_", "-"))

MODEL = "openjev/OpenJev-Flash-9B"
MAX_LENGTH = 8192 if MODE == "few_shot" else 2048
THRESHOLD = 0.5
TUNE_ON_DEV = False
DEV = ""  # For final evaluation: DATA=test, DEV=val, TUNE_ON_DEV=True

options = dict(data=DATA, output=OUTPUT, dev=DEV if TUNE_ON_DEV else "",
               model=MODEL, max_length=MAX_LENGTH)
if MODE == "zero_shot":
    job = run.ZeroShot(**options)
elif MODE == "few_shot":
    job = run.FewShot(train=TRAIN, shots=SHOTS, **options)
else:
    raise ValueError("MODE must be zero_shot or few_shot.")


## 3. Kiểm tra input
Kiểm tra GPU BF16, tokenizer và độ dài từng prompt trước khi tải weights. Không tự cắt title, description hoặc comment. Nếu vượt MAX_LENGTH, tăng giới hạn theo thông báo rồi tạo job và chạy check lại.


In [ ]:
job.check()


## 4. Chạy inference
Mỗi comment có 9 câu yes/no cho identity và 1 câu chọn scope. Gold của DATA chỉ dùng để tính metrics; không được đưa vào prompt. Few-shot chỉ dùng gold của ví dụ train để trả lời chữ A/B trong các lượt minh họa. Không identity nào vượt ngưỡng thì ghi abstain và giữ mẫu trong đánh giá.

Xác suất là restricted A/B softmax, chưa calibration cho dataset này và không phải số benchmark của API OpenJev gốc. Bản fine-tune mặc định chọn ngưỡng trên dev; để so sánh cùng protocol, dùng DATA=test và bật TUNE_ON_DEV với cùng dev cho từng model. Ngưỡng cố định 0.5 là baseline không tune bằng nhãn. Few-shot có thể giúp hoặc làm giảm chất lượng; cần chạy thực nghiệm mới kết luận.


In [ ]:
metrics = job.infer(threshold=THRESHOLD, tune_on_dev=TUNE_ON_DEV)
print("Results:", OUTPUT)


## 5. Tải kết quả
Chỉ lưu settings.json (model revision, câu hỏi, mode, hash train nếu dùng, ID và phân bố nhãn của ví dụ), predictions.csv và metrics.json. Không lưu nội dung comment, bản sao dataset, adapter hoặc checkpoint training. Tải ZIP trước khi session kết thúc. Đổi MODE rồi chạy lại từ phần 2 để so sánh; chọn OUTPUT mới nếu thư mục đó đã chứa kết quả.


In [ ]:
import zipfile

folder = Path(OUTPUT).resolve()
archive = folder.parent / (folder.name + ".zip")
if not (folder / "metrics.json").is_file():
    raise FileNotFoundError("Run inference successfully before exporting.")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in ("settings.json", "predictions.csv", "metrics.json"):
        z.write(folder / name, arcname=name)
print("ZIP:", archive)

try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
else:
    files.download(str(archive))
